In [1]:
%load_ext autoreload

In [2]:
%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
%autoreload 2

In [4]:
import sys
sys.path.append('../src/')

In [5]:
import geocode 

In [6]:
import openmeteo_requests

import sys
import pandas as pd
import geopandas as gpd
import fiona
import requests_cache
from retry_requests import retry
import datetime

from geocode import geocoded_cities_pipeline


In [7]:
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"

In [8]:
cities_gdf = geocoded_cities_pipeline()


In [9]:
cities_gdf.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [10]:
df_cloud_cities_data_full = pd.DataFrame(columns=['date', 'cloud_cover'])
dict_cloud_cities_data_full = {}


In [12]:
results_list = []

for idx in cities_gdf.index:
    
    if idx ==4:
        break
    else:

        print("idx")

        print(idx)

        geom = cities_gdf.at[idx, 'geometry']
        #print(geom)

        #print(cities_gdf.at[idx, 'city_name'])

        x_coord = geom.x
        #print(x_coord)

        y_coord = geom.y
        #print(y_coord)

        params = {

            "latitude": y_coord,    # y coord
            "longitude": x_coord,  # x coord
            "start_date": "2022-10-04",
            "end_date": "2022-10-10",
            "hourly": ["cloud_cover"],
        }
        responses = openmeteo.weather_api(url, params=params)

        # Process first location. Add a for-loop for multiple locations or weather models
        response = responses[0]
        # print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
        # print(f"Elevation: {response.Elevation()} m asl")
        # print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

        # Process hourly data. The order of variables needs to be the same as requested.  ##### XXX Note that the .Daily endpoint of the open-meteo API does NOT return cloud_cover information
        hourly = response.Hourly()
        print(hourly)
        hourly_cloud_cover = hourly.Variables(0).ValuesAsNumpy()
    #     hourly_cloud_cover_low = hourly.Variables(1).ValuesAsNumpy()
    #     hourly_cloud_cover_mid = hourly.Variables(2).ValuesAsNumpy()
    #     hourly_cloud_cover_high = hourly.Variables(3).ValuesAsNumpy()

        hourly_data = {"date": pd.date_range(
            start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
            end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
            freq = pd.Timedelta(seconds = hourly.Interval()),
            inclusive = "left"
        )}

        hourly_data["cloud_cover"] = hourly_cloud_cover
    #     hourly_data["cloud_cover_low"] = hourly_cloud_cover_low
    #     hourly_data["cloud_cover_mid"] = hourly_cloud_cover_mid
    #     hourly_data["cloud_cover_high"] = hourly_cloud_cover_high

        hourly_dataframe_temp = pd.DataFrame(data = hourly_data)

        #print("\nHourly data\n", hourly_dataframe_temp)

        df_datetime_temp = hourly_dataframe_temp
        df_cloud_temp = df_datetime_temp.set_index('date').resample(rule='24h').mean()
        df_cloud_temp = df_cloud_temp.reset_index()
#         print("before transpose")
#         print(df_cloud_temp)
        df_cloud_temp['date'] = df_cloud_temp['date'].dt.strftime('%Y-%m-%d')
        #df_cloud_temp['date'] = df_cloud_temp['date'].date().strftime('%Y-%m-%d')
        #print(df_cloud_temp)
        df_cloud_temp = df_cloud_temp.transpose()
        print(df_cloud_temp)
        #df_cloud_temp = df_cloud_temp.set_index(df_cloud_temp.columns[0]).transpose()
        

    #       df_cities_temp = cities_gdf.head(idx +1)
        city_row = cities_gdf.loc[[idx]]
        print(city_row['city_name'])

        #print(temp_df)

        df_merged_temp = pd.merge(city_row, df_cloud_temp, how="cross")
        results_list.append(df_merged_temp)

        print(df_merged_temp)

        #hourly_dict = hourly_dataframe.to_dict('index')
   

final_df = pd.concat(results_list, ignore_index=True)
print("\nFinal merged dataframe:")

print(final_df)


idx
0
                      0           1           2           3           4  \
date         2022-10-04  2022-10-05  2022-10-06  2022-10-07  2022-10-08   
cloud_cover   77.333336       100.0   95.833336   96.708336       88.25   

                      5           6  
date         2022-10-09  2022-10-10  
cloud_cover   99.291664   78.958336  
0    Tokyo
Name: city_name, dtype: object
  city_name country adm0name   pop_max                    geometry  \
0     Tokyo   Japan    Japan  35676000  POINT (139.74946 35.68696)   
1     Tokyo   Japan    Japan  35676000  POINT (139.74946 35.68696)   

            0           1           2           3           4           5  \
0  2022-10-04  2022-10-05  2022-10-06  2022-10-07  2022-10-08  2022-10-09   
1   77.333336       100.0   95.833336   96.708336       88.25   99.291664   

            6  
0  2022-10-10  
1   78.958336  
idx
1
                      0           1           2           3           4  \
date         2022-10-04  2022-10-05  202